# Ridge versus RBF-SVR

**Purpose:** Compare linear and nonlinear regressors on matched W-ALL9 and D-ALL9 inputs.

**Required inputs:** Prepared Baček HDF5; 04 foundational results ZIP.

**Outputs:** Ridge/SVR comparison and timing records.

**Usage:** Run after 04. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, experiment definitions and helper functions
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from threadpoolctl import threadpool_info

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
OUT = Path('09_Bacek_Ridge_vs_RBF_SVR_Comparison_results')
OUTPUT_ZIP = Path('09_Bacek_Ridge_vs_RBF_SVR_Comparison_results.zip')
SOURCE_SHA256 = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
REFERENCE_SHA256 = 'd0d5fb49f3f767dc1b99c790afda9aad630bfae5199fced5c5512c14329c7752'
CHANNELS = ('hip_left','hip_right','knee_left','knee_right','ankle_left',
            'ankle_right','pelvis_sagittal','pelvis_frontal','pelvis_transverse')
OUTER_FOLDS = {
    1: ('S08','S10','S16','S21'),
    2: ('S11','S13','S17','S20'),
    3: ('S01','S02','S05','S18'),
    4: ('S03','S07','S12','S14','S19'),
    5: ('S04','S06','S09','S15'),
}
FOLD_OF = {p: k for k, ps in OUTER_FOLDS.items() for p in ps}
TRIAL_COUNTS = (55,53,56,65,55)
REPS = ('W-ALL9','D-ALL9')
ALPHAS = (0.01,0.1,1.0,10.0,100.0,1000.0)
C_GRID = (0.1,1.0,10.0)
EPSILON_GRID = (0.05,0.2)
SVR_GRID = tuple((c,e) for c in C_GRID for e in EPSILON_GRID)
PROCEDURES = ('W-ALL9_RIDGE','W-ALL9_SVR','D-ALL9_RIDGE','D-ALL9_SVR')


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    result = hashlib.sha256()
    with path.open('rb') as fh:
        for block in iter(lambda: fh.read(8*1024*1024), b''):
            result.update(block)
    return result.hexdigest()


def read_zip_csv(archive, name):
    with archive.open(name) as fh:
        return list(csv.DictReader(io.TextIOWrapper(fh, encoding='utf-8')))


def save_csv(path, records, fields):
    with path.open('w', encoding='utf-8', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)


def rmse(y, pred):
    err = np.asarray(pred, dtype=np.float64) - np.asarray(y, dtype=np.float64)
    return float(np.sqrt(np.mean(err**2, dtype=np.float64)))


def metrics(y, pred):
    y = np.asarray(y, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    err = pred-y
    sst = float(np.sum((y-y.mean())**2, dtype=np.float64))
    return {'n_trials': int(len(y)), 'RMSE_W_per_kg': rmse(y,pred),
            'MAE_W_per_kg': float(np.mean(np.abs(err), dtype=np.float64)),
            'bias_W_per_kg': float(np.mean(err, dtype=np.float64)),
            'R2': float(1-np.sum(err**2, dtype=np.float64)/sst) if sst > 0 else None}


def make_model(family, rep, *, alpha=None, c=None, epsilon=None):
    check(family in ('RIDGE','SVR') and rep in REPS, 'Unknown estimator or representation')
    if family == 'RIDGE':
        check(alpha in ALPHAS, 'Invalid Ridge alpha')
        model = Ridge(alpha=alpha, fit_intercept=True)
    else:
        check((c,epsilon) in SVR_GRID, 'Invalid SVR configuration')
        model = SVR(kernel='rbf', gamma='scale', C=c, epsilon=epsilon)
    return Pipeline([('scale', StandardScaler()), ('estimator', model)])


def choose_svr(rows):
    # Exact floating-point tie: smaller C, then LARGER epsilon.
    return min(rows, key=lambda r: (r['inner_pooled_RMSE_W_per_kg'],
                                    r['C'], -r['epsilon']))


def single_trial_prediction_times(model, x, reference_predictions):
    elapsed = []
    values = []
    for row in x:
        start = time.perf_counter()
        value = float(model.predict(row.reshape(1,-1))[0])
        elapsed.append(time.perf_counter()-start)
        values.append(value)
    check(np.allclose(values, reference_predictions, rtol=1e-10, atol=1e-10),
          'Single-trial predictions differ from the batch predictions')
    return elapsed


print('Execution environment:', platform.python_version(),
      'NumPy', np.__version__, 'scikit-learn', sklearn.__version__)
check(len(SVR_GRID) == 6 and len(FOLD_OF) == 21 and len(PROCEDURES) == 4,
      'Experimental candidate or fold definitions are invalid')

In [ ]:
# 2 — Verify Notebook 04 evidence, including exact participant partitions and Ridge selections
check(REFERENCE.is_file(), f'Cannot find required reference ZIP: {REFERENCE.name}')
reference_sha = sha256_file(REFERENCE)
check(reference_sha == REFERENCE_SHA256, 'Notebook 04 ZIP SHA-256 mismatch: stop and reconcile')
with zipfile.ZipFile(REFERENCE) as archive:
    ref_summary = json.loads(archive.read('run_summary.json'))
    ref_ledger = read_zip_csv(archive, 'eligible_trial_ledger.csv')
    ref_outer = read_zip_csv(archive, 'outer_fold_manifest.csv')
    ref_inner = read_zip_csv(archive, 'inner_fold_manifest.csv')
    ref_inner_scores = read_zip_csv(archive, 'inner_candidate_scores.csv')
    ref_pooled_scores = read_zip_csv(archive, 'pooled_candidate_scores.csv')
    ref_selected = read_zip_csv(archive, 'selected_models.csv')
    ref_pred = read_zip_csv(archive, 'held_out_predictions.csv')
    ref_timings = read_zip_csv(archive, 'outer_model_timings.csv')
    ref_checks = read_zip_csv(archive, 'verification_checks.csv')

check(ref_summary['status'] == 'PASS' and ref_summary['source_sha256'] == SOURCE_SHA256,
      'Reference metadata verification failed')
check(len(ref_ledger) == 284 and len(ref_outer) == 5 and len(ref_inner) == 20,
      'Missing reference eligible trials or fold records')
check(all(r['status']=='PASS' for r in ref_checks), 'Reference contains failed checks')
uid_ref = {r['trial_uid']:r for r in ref_ledger}
check(len(uid_ref) == 284 and len({r['subject_id'] for r in ref_ledger}) == 21
      and sum(int(r['retained_cycles']) for r in ref_ledger) == 60757,
      'Reference population totals invalid')
check(all(int(r['outer_fold']) == FOLD_OF[r['subject_id']] and
          np.isfinite(float(r['observed_W_per_kg'])) and float(r['observed_W_per_kg'])>0
          for r in ref_ledger), 'Invalid source-trial identities or targets in reference')

inner_indices = {}
for outer in OUTER_FOLDS:
    r = [t for t in ref_outer if int(t['outer_fold']) == outer]
    check(len(r)==1 and set(r[0]['test_participants'].split(';'))==set(OUTER_FOLDS[outer])
          and set(r[0]['development_participants'].split(';'))==set(FOLD_OF)-set(OUTER_FOLDS[outer])
          and int(r[0]['n_test_trials']) == TRIAL_COUNTS[outer-1],
          f'Incorrect reference outer fold {outer}')
    part = [t for t in ref_inner if int(t['outer_fold']) == outer]
    check(len(part)==4 and {int(t['inner_fold']) for t in part}==set(range(1,5)),
          f'Inner folds incomplete for outer {outer}')
    assigned = []
    for t in part:
        inner = int(t['inner_fold'])
        train = set(t['train_participants'].split(';'))
        val = set(t['validation_participants'].split(';'))
        test = set(t['outer_test_participants'].split(';'))
        check(not(train & val or train & test or val & test) and
              train | val == set(FOLD_OF)-test and test == set(OUTER_FOLDS[outer]),
              f'Inner participant leakage in outer {outer}/inner {inner}')
        check(sum(uid_ref[u]['subject_id'] in train for u in uid_ref) == int(t['n_train_trials']) and
              sum(uid_ref[u]['subject_id'] in val for u in uid_ref) == int(t['n_validation_trials']),
              f'Incorrect inner trial counts for outer {outer}/inner {inner}')
        inner_indices[outer,inner] = (train,val)
        assigned.extend(val)
    check(len(assigned)==len(set(assigned)) and set(assigned)==set(FOLD_OF)-set(OUTER_FOLDS[outer]),
          f'Inner validation does not partition development participants in outer {outer}')

ref_ridge_predictions = {}
ref_ridge_selected = {}
for rep in REPS:
    procedure04 = {'W-ALL9':'W-ALL9_FIXED','D-ALL9':'D-ALL9_FIXED'}[rep]
    p = {r['trial_uid']:r for r in ref_pred if r['procedure']==procedure04}
    s = {int(r['outer_fold']):r for r in ref_selected if r['procedure']==procedure04}
    scores = [r for r in ref_pooled_scores if r['representation']==rep]
    detailed = [r for r in ref_inner_scores if r['representation']==rep]
    check(len(p)==284 and set(p)==set(uid_ref) and len(s)==5 and len(scores)==30
          and len(detailed)==120, f'Reference Ridge {rep} evidence incomplete')
    for uid,r in p.items():
        check(r['subject_id']==uid_ref[uid]['subject_id'] and
              int(r['outer_fold'])==int(uid_ref[uid]['outer_fold']) and
              float(r['observed_W_per_kg'])==float(uid_ref[uid]['observed_W_per_kg'])
              and np.isfinite(float(r['predicted_W_per_kg'])),
              f'Reference Ridge prediction mismatch: {uid}')
    for outer in OUTER_FOLDS:
        subset = [r for r in scores if int(r['outer_fold'])==outer]
        check(len(subset)==6 and {float(r['alpha']) for r in subset}==set(ALPHAS),
              f'Reference Ridge candidates incomplete: {rep}, fold {outer}')
        best = min(subset, key=lambda r:(float(r['inner_pooled_RMSE_W_per_kg']),float(r['alpha'])))
        chosen = s[outer]
        check(float(chosen['alpha']) == float(best['alpha']) and
              float(chosen['inner_pooled_RMSE_W_per_kg']) == float(best['inner_pooled_RMSE_W_per_kg']),
              f'Reference Ridge selection mismatch for {rep} fold {outer}')
    ref_ridge_predictions[rep] = p
    ref_ridge_selected[rep] = s

print('Notebook 04 SHA-256, trial population, 20 saved inner partitions and all Ridge references verified')

In [ ]:
# 3 — Synthetic unit checks: W/D mathematics, exact SVR ties and fit-only scaling

def trial_features(trial):
    means=[]
    n_cycles=None
    for channel in CHANNELS:
        check(channel in trial, f'Missing gait channel: {channel}')
        array=np.asarray(trial[channel][()], dtype=np.float64)
        check(array.ndim==2 and array.shape[1]==101 and np.isfinite(array).all(),
              f'Invalid angle data for {channel}')
        if n_cycles is None:
            n_cycles=array.shape[0]
        check(array.shape[0]==n_cycles,'Channel cycle count mismatch')
        means.append(array.mean(axis=0, dtype=np.float64))
    m=np.stack(means, axis=0)
    w=m.reshape(909)
    d=[]
    for curve in m:
        for q in range(4):
            d.append(float(np.trapezoid(curve[25*q:25*q+26], dx=1.0)/25.0))
        d.append(float(np.std(curve, ddof=0)))
        d.append(float(np.max(curve)-np.min(curve)))
    return w, np.asarray(d,dtype=np.float64), n_cycles

synthetic_trial={name: np.tile(np.linspace(0,100,101)+i, (3,1))
                 for i,name in enumerate(CHANNELS)}
s_w,s_d,s_n=trial_features(synthetic_trial)
check(s_w.shape==(909,) and s_d.shape==(54,) and s_n==3,
      'Synthetic W/D feature dimensions failed')
check(np.allclose(s_d[:4],[12.5,37.5,62.5,87.5], atol=1e-12,rtol=0)
      and abs(s_d[5]-100.0)<1e-12 and abs(s_w[101]-1.0)<1e-12,
      'Trapezoidal integration or channel order incorrect')
check((choose_svr([{'inner_pooled_RMSE_W_per_kg':0.5,'C':c,'epsilon':e}
                   for c,e in SVR_GRID])['C'],
       choose_svr([{'inner_pooled_RMSE_W_per_kg':0.5,'C':c,'epsilon':e}
                   for c,e in SVR_GRID])['epsilon']) == (0.1,0.2),
      'SVR exact-tie rule failed')

# Synthetic scaling check: validation/test data must not influence the fitted scaler.
rand=np.random.default_rng(2026)
a=rand.normal(size=(30,54)); b=np.full((5,54),1000.0)
for family,params in [('RIDGE',dict(alpha=1.0)),('SVR',dict(c=1.0,epsilon=0.05))]:
    model=make_model(family,'D-ALL9',**params)
    model.fit(a,rand.normal(size=30))
    check(np.allclose(model.named_steps['scale'].mean_,a.mean(axis=0),atol=1e-12)
          and np.isfinite(model.predict(b)).all(),
          f'Training-only scaling or inference failed for {family}')
print('Synthetic feature, SVR tie-rule and training-only scaling checks passed')

In [ ]:
# 4 — Read the corrected Baček data locally and match every W/D row to Notebook 04
check(H5.is_file(), f'Missing corrected Baček dataset: {H5.name}')
source_sha=sha256_file(H5)
check(source_sha == SOURCE_SHA256,'Corrected Baček source fingerprint mismatch')
rows=[]; w_rows=[]; d_rows=[]
with h5py.File(H5, 'r') as h5:
    check(set(h5['subjects'])==set(FOLD_OF),'Unexpected source participants')
    for person in sorted(h5['subjects']):
        for trial_id in sorted(h5['subjects'][person]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1<=int(trial_id[1:])<=15):
                continue
            trial=h5['subjects'][person]['trials'][trial_id]
            uid=f'{person}_{trial_id}'
            w,d,n=trial_features(trial)
            for attr in ('valid','comparison_eligible','unrestricted_comparison_eligible'):
                check(attr in trial.attrs and np.asarray(trial.attrs[attr]).reshape(-1)[0]==1,
                      f'Invalid prepared record flag: {uid}/{attr}')
            check('metcost_mean' in trial and 'original_cycle_number' in trial,
                  f'Missing trial target or cycle provenance: {uid}')
            target=np.asarray(trial['metcost_mean'][()])
            check(target.size==1 and np.issubdtype(target.dtype,np.number)
                  and np.isrealobj(target), f'Invalid target type: {uid}')
            yval=float(target.reshape(-1)[0])
            check(np.isfinite(yval) and yval>0, f'Invalid target value: {uid}')
            indices=np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(indices)==n and np.isfinite(indices).all() and
                  np.all(indices==np.floor(indices)) and np.all(np.diff(indices)>0),
                  f'Invalid retained cycle indices: {uid}')
            if uid == 'S06_T11':
                check(n==85 and uid not in uid_ref, 'Sensitivity trial has unexpected status')
                continue
            check(n>=100 and uid in uid_ref, f'Unexpected eligible trial: {uid}')
            record=uid_ref[uid]
            check(person==record['subject_id'] and trial_id==record['trial_id'] and
                  FOLD_OF[person]==int(record['outer_fold']) and
                  n==int(record['retained_cycles']) and
                  yval==float(record['observed_W_per_kg']),
                  f'Trial identity, cycle count or stored target differs from Notebook 04: {uid}')
            rows.append({'trial_uid':uid,'subject_id':person,'trial_id':trial_id,
                         'outer_fold':FOLD_OF[person],'retained_cycles':n,
                         'observed_W_per_kg':yval})
            w_rows.append(w);d_rows.append(d)

check(len(rows)==284 and len({r['trial_uid'] for r in rows})==284 and
      set(r['trial_uid'] for r in rows)==set(uid_ref) and
      sum(r['retained_cycles'] for r in rows)==60757,
      'Incorrect primary trial population')
X={'W-ALL9':np.stack(w_rows),'D-ALL9':np.stack(d_rows)}
y=np.asarray([r['observed_W_per_kg'] for r in rows],dtype=np.float64)
groups=np.asarray([r['subject_id'] for r in rows])
uids=np.asarray([r['trial_uid'] for r in rows])
check(X['W-ALL9'].shape==(284,909) and X['D-ALL9'].shape==(284,54)
      and np.isfinite(X['W-ALL9']).all() and np.isfinite(X['D-ALL9']).all(),
      'Incorrect or invalid W/D feature matrices')
# Reconstruct exact individual-trial index lists from saved participant assignments.
outer_idx={}
inner_idx={}
for outer, people in OUTER_FOLDS.items():
    test=np.flatnonzero(np.isin(groups,people))
    dev=np.flatnonzero(~np.isin(groups,people))
    check(len(test)==TRIAL_COUNTS[outer-1] and len(dev)+len(test)==284,
          f'Outer trial count mismatch: {outer}')
    outer_idx[outer]=(dev,test)
    for inner in (1,2,3,4):
        train_people,val_people=inner_indices[outer,inner]
        tr=np.flatnonzero(np.isin(groups,list(train_people)))
        va=np.flatnonzero(np.isin(groups,list(val_people)))
        check(len(tr)+len(va)==len(dev) and
              len(set(groups[tr]) & set(groups[va]))==0 and
              len(set(groups[test]) & (set(groups[tr])|set(groups[va])))==0,
              'Saved participant-separation boundary violated')
        inner_idx[outer,inner]=(tr,va)
print('Matched numeric source and Notebook 04:',len(rows),'trials, W',X['W-ALL9'].shape,
      'D',X['D-ALL9'].shape)

In [ ]:
# 5 — Development-only SVR selection: six candidates per representation/outer fold
started_at=datetime.now(timezone.utc).isoformat()
experiment_start=time.perf_counter()
inner_score_rows=[]
pooled_score_rows=[]
svr_selected={}

for outer in OUTER_FOLDS:
    for rep in REPS:
        scores=[]
        for c,epsilon in SVR_GRID:
            total_sq=0.0
            n_val=0
            for inner in (1,2,3,4):
                train,val=inner_idx[outer,inner]
                model=make_model('SVR',rep,c=c,epsilon=epsilon)
                begin=time.perf_counter()
                model.fit(X[rep][train],y[train])
                fit_seconds=time.perf_counter()-begin
                pred_train=model.predict(X[rep][train])
                pred_val=model.predict(X[rep][val])
                check(np.isfinite(pred_train).all() and np.isfinite(pred_val).all(),
                      'Nonfinite SVR inner prediction')
                residual=np.asarray(pred_val-y[val],dtype=np.float64)
                total_sq+=float(np.sum(residual**2,dtype=np.float64))
                n_val+=len(val)
                inner_score_rows.append({
                    'outer_fold':outer,'inner_fold':inner,'estimator':'SVR',
                    'representation':rep,'alpha':'','C':c,'epsilon':epsilon,'gamma':'scale',
                    'n_features':X[rep].shape[1],
                    'n_train_trials':len(train),'n_validation_trials':len(val),
                    'train_RMSE_W_per_kg':rmse(y[train],pred_train),
                    'validation_RMSE_W_per_kg':rmse(y[val],pred_val),
                    'fit_seconds':fit_seconds,'source_run':'NOTEBOOK09'})
            check(n_val==len(outer_idx[outer][0]), 'Inner validation does not cover development trials')
            score={'outer_fold':outer,'estimator':'SVR','representation':rep,
                   'alpha':'','C':c,'epsilon':epsilon,'gamma':'scale',
                   'n_features':X[rep].shape[1],
                   'inner_pooled_RMSE_W_per_kg':float(np.sqrt(total_sq/n_val)),
                   'source_run':'NOTEBOOK09'}
            scores.append(score)
            pooled_score_rows.append(score)
        svr_selected[outer,rep]=choose_svr(scores)
    print(f'Outer {outer}: SVR selected W {svr_selected[outer,"W-ALL9"]["C"]}/'
          f'{svr_selected[outer,"W-ALL9"]["epsilon"]}; D '
          f'{svr_selected[outer,"D-ALL9"]["C"]}/'
          f'{svr_selected[outer,"D-ALL9"]["epsilon"]}')

# Reuse exactly the previous Ridge development scores, not an SVR-dependent re-selection.
for r in ref_inner_scores:
    if r['representation'] not in REPS:
        continue
    inner_score_rows.append({
        'outer_fold':int(r['outer_fold']),'inner_fold':int(r['inner_fold']),
        'estimator':'RIDGE','representation':r['representation'],
        'alpha':float(r['alpha']),'C':'','epsilon':'','gamma':'',
        'n_features':int(r['n_features']),
        'n_train_trials':int(r['n_train_trials']),
        'n_validation_trials':int(r['n_validation_trials']),
        'train_RMSE_W_per_kg':float(r['train_RMSE_W_per_kg']),
        'validation_RMSE_W_per_kg':float(r['validation_RMSE_W_per_kg']),
        'fit_seconds':float(r['fit_seconds']),'source_run':'NOTEBOOK04'})
for r in ref_pooled_scores:
    if r['representation'] not in REPS:
        continue
    pooled_score_rows.append({
        'outer_fold':int(r['outer_fold']),'estimator':'RIDGE',
        'representation':r['representation'],'alpha':float(r['alpha']),
        'C':'','epsilon':'','gamma':'',
        'n_features':int(r['n_features']),
        'inner_pooled_RMSE_W_per_kg':float(r['inner_pooled_RMSE_W_per_kg']),
        'source_run':'NOTEBOOK04'})
check(len(inner_score_rows)==480 and len(pooled_score_rows)==120
      and len(svr_selected)==10,'Unexpected candidate score or selected-model count')

In [ ]:
# 6 — Held-out SVR prediction, exact Ridge reuse and same-session RC-4 timing comparison
prediction_rows=[]
selected_rows=[]
timing_rows=[]
timing_prediction_difference=[]

for outer in OUTER_FOLDS:
    dev,test=outer_idx[outer]
    for rep in REPS:
        for family in ('RIDGE','SVR'):
            if family=='RIDGE':
                old=ref_ridge_selected[rep][outer]
                alpha=float(old['alpha'])
                selected_score=float(old['inner_pooled_RMSE_W_per_kg'])
                model=make_model('RIDGE',rep,alpha=alpha)
                c=epsilon=''
                source='NOTEBOOK04'
            else:
                chosen=svr_selected[outer,rep]
                c=float(chosen['C']);epsilon=float(chosen['epsilon']);alpha=''
                selected_score=float(chosen['inner_pooled_RMSE_W_per_kg'])
                model=make_model('SVR',rep,c=c,epsilon=epsilon)
                source='NOTEBOOK09'
            procedure=f'{rep}_{family}'
            selected_rows.append({'outer_fold':outer,'procedure':procedure,
                                  'estimator':family,'representation':rep,
                                  'alpha':alpha,'C':c,'epsilon':epsilon,
                                  'gamma':'scale' if family=='SVR' else '',
                                  'n_features':X[rep].shape[1],
                                  'inner_pooled_RMSE_W_per_kg':selected_score,
                                  'source_run':source})
            start=time.perf_counter()
            model.fit(X[rep][dev],y[dev])
            fit_seconds=time.perf_counter()-start
            start=time.perf_counter()
            new_preds=np.asarray(model.predict(X[rep][test]),dtype=np.float64)
            batch_seconds=time.perf_counter()-start
            check(new_preds.shape==(len(test),) and np.isfinite(new_preds).all(),
                  f'Invalid {procedure} prediction values')
            if family=='RIDGE':
                # Only time this refit; the scientific result comes from Notebook 04.
                official_preds=np.asarray([float(ref_ridge_predictions[rep][uids[i]]['predicted_W_per_kg'])
                                           for i in test],dtype=np.float64)
                max_abs_difference=float(np.max(np.abs(new_preds-official_preds)))
                check(np.allclose(new_preds,official_preds,rtol=1e-8,atol=1e-8),
                      f'Timing-only Ridge refit unexpectedly differs from Notebook 04: {rep}, {outer}')
                timing_prediction_difference.append({'outer_fold':outer,'representation':rep,
                                                     'max_abs_difference_W_per_kg':max_abs_difference})
            else:
                official_preds=new_preds
                max_abs_difference=0.0
            single_times=single_trial_prediction_times(model,X[rep][test],new_preds)
            timing_rows.append({'outer_fold':outer,'estimator':family,'representation':rep,
                                'alpha':alpha,'C':c,'epsilon':epsilon,
                                'n_train_trials':len(dev),'n_predicted_trials':len(test),
                                'fit_seconds':fit_seconds,
                                'batch_prediction_seconds':batch_seconds,
                                'batch_amortised_seconds_per_trial':batch_seconds/len(test),
                                'single_trial_prediction_total_seconds':float(np.sum(single_times)),
                                'single_trial_prediction_mean_seconds':float(np.mean(single_times)),
                                'single_trial_prediction_median_seconds':float(np.median(single_times)),
                                'ridge_timing_refit_max_abs_difference_W_per_kg':
                                    max_abs_difference if family=='RIDGE' else '',
                                'run_type':'TIMING_ONLY_REFIT' if family=='RIDGE' else 'SVR_EXPERIMENT'})
            for local_idx,i in enumerate(test):
                prediction_rows.append({'procedure':procedure,'estimator':family,
                                        'representation':rep,'outer_fold':outer,
                                        'subject_id':rows[i]['subject_id'],
                                        'trial_id':rows[i]['trial_id'],'trial_uid':uids[i],
                                        'observed_W_per_kg':float(y[i]),
                                        'predicted_W_per_kg':float(official_preds[local_idx]),
                                        'source_run':source})
    print(f'Outer {outer}: completed W/D Ridge and SVR prediction ledgers and timing comparisons')

experiment_seconds=time.perf_counter()-experiment_start
print('Completed E4.1 modelling in',round(experiment_seconds,2),'seconds')

In [ ]:
# 7 — Verify complete paired ledgers and compute pooled/fold metrics and contrasts
checks=[]
def verify(name, condition):
    ok=bool(condition)
    checks.append({'check':name,'status':'PASS' if ok else 'FAIL'})
    check(ok,name)

verify('authoritative_source_and_reference_fingerprints',
       source_sha==SOURCE_SHA256 and reference_sha==REFERENCE_SHA256)
verify('exact_primary_trial_population',len(rows)==284 and len(set(uids))==284
       and sum(r['retained_cycles'] for r in rows)==60757)
verify('exact_outer_and_inner_partition_records',len(ref_outer)==5 and len(ref_inner)==20
       and len(outer_idx)==5 and len(inner_idx)==20)
verify('correct_svr_candidate_count',len([r for r in pooled_score_rows if r['estimator']=='SVR'])==60)
verify('correct_ridge_reused_candidate_count',len([r for r in pooled_score_rows if r['estimator']=='RIDGE'])==60)
verify('all_480_inner_candidate_records',len(inner_score_rows)==480)
verify('all_20_selected_models',len(selected_rows)==20)
verify('all_1136_held_out_predictions',len(prediction_rows)==284*4)
verify('unique_procedure_trial_keys',len({(r['procedure'],r['trial_uid']) for r in prediction_rows})==len(prediction_rows))
verify('complete_matched_284_trial_sets',all(
       {r['trial_uid'] for r in prediction_rows if r['procedure']==p}==set(uids)
       for p in PROCEDURES))
verify('predictions_match_target_and_group_ledger',all(
       int(r['outer_fold'])==int(uid_ref[r['trial_uid']]['outer_fold']) and
       r['subject_id']==uid_ref[r['trial_uid']]['subject_id'] and
       r['observed_W_per_kg']==float(uid_ref[r['trial_uid']]['observed_W_per_kg']) and
       np.isfinite(r['predicted_W_per_kg']) for r in prediction_rows))
verify('reused_ridge_predictions_are_identical',all(
       r['predicted_W_per_kg']==float(ref_ridge_predictions[r['representation']][r['trial_uid']]['predicted_W_per_kg'])
       for r in prediction_rows if r['estimator']=='RIDGE'))
verify('selected_svr_candidates_minimise_inner_rmse',all(
       svr_selected[outer,rep]==choose_svr([r for r in pooled_score_rows
           if r['estimator']=='SVR' and int(r['outer_fold'])==outer and r['representation']==rep])
       for outer in OUTER_FOLDS for rep in REPS))
verify('timing_records_cover_every_model_representation_and_fold',
       len(timing_rows)==20 and len({(r['outer_fold'],r['estimator'],r['representation'])
                                     for r in timing_rows})==20)
verify('valid_positive_computational_timing_records',all(
       r['fit_seconds']>=0 and r['batch_prediction_seconds']>=0
       and r['single_trial_prediction_mean_seconds']>=0 for r in timing_rows))
verify('ten_ridge_refits_consistent_with_reference',len(timing_prediction_difference)==10)

pooled_metrics=[]
fold_metrics=[]
for procedure in PROCEDURES:
    subset=[r for r in prediction_rows if r['procedure']==procedure]
    pooled_metrics.append({'procedure':procedure,**metrics(
        [r['observed_W_per_kg'] for r in subset],
        [r['predicted_W_per_kg'] for r in subset])})
    for outer in OUTER_FOLDS:
        fold_subset=[r for r in subset if r['outer_fold']==outer]
        fold_metrics.append({'outer_fold':outer,'procedure':procedure,**metrics(
            [r['observed_W_per_kg'] for r in fold_subset],
            [r['predicted_W_per_kg'] for r in fold_subset])})
verify('all_model_metrics_valid',len(pooled_metrics)==4 and len(fold_metrics)==20
       and all(np.isfinite(r['RMSE_W_per_kg']) for r in pooled_metrics+fold_metrics))

metric_by_proc={r['procedure']:r['RMSE_W_per_kg'] for r in pooled_metrics}
paired_diffs=[]
for rep in REPS:
    ridge=metric_by_proc[f'{rep}_RIDGE']
    svr=metric_by_proc[f'{rep}_SVR']
    paired_diffs.append({'representation':rep,'ridge_RMSE_W_per_kg':ridge,
                         'svr_RMSE_W_per_kg':svr,
                         'ridge_minus_svr_RMSE_W_per_kg':ridge-svr})
interaction=(paired_diffs[0]['ridge_minus_svr_RMSE_W_per_kg']-
             paired_diffs[1]['ridge_minus_svr_RMSE_W_per_kg'])
print('\nE4.1 pooled trial-weighted RMSE (W/kg):')
for r in pooled_metrics:
    print(f"  {r['procedure']:18s} {r['RMSE_W_per_kg']:.6f}")
print('Ridge minus SVR improvements:',[(r['representation'],round(r['ridge_minus_svr_RMSE_W_per_kg'],6))
                                      for r in paired_diffs])
print('W minus D improvement difference (descriptive):',round(interaction,6))
print('Verification checks passed:',len(checks),'/',len(checks))

In [ ]:
# 8 — Archive run evidence, source fingerprints and measured computational environment
OUT.mkdir(parents=True,exist_ok=True)
notebook_file=Path('09_Bacek_Ridge_vs_RBF_SVR_Comparison.ipynb')
notebook_code_sha=None
if notebook_file.is_file():
    doc=json.loads(notebook_file.read_text(encoding='utf-8'))
    code_text=''.join(''.join(c['source']) for c in doc['cells'] if c['cell_type']=='code')
    notebook_code_sha=hashlib.sha256(code_text.encode('utf-8')).hexdigest()

save_csv(OUT/'eligible_trial_ledger.csv',rows,
         ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles','observed_W_per_kg'])
save_csv(OUT/'inner_fold_manifest.csv',ref_inner,
         ['outer_fold','inner_fold','train_participants','validation_participants',
          'outer_test_participants','n_train_trials','n_validation_trials'])
save_csv(OUT/'inner_candidate_scores.csv',inner_score_rows,
         ['outer_fold','inner_fold','estimator','representation','alpha','C','epsilon','gamma',
          'n_features','n_train_trials','n_validation_trials',
          'train_RMSE_W_per_kg','validation_RMSE_W_per_kg','fit_seconds','source_run'])
save_csv(OUT/'pooled_candidate_scores.csv',pooled_score_rows,
         ['outer_fold','estimator','representation','alpha','C','epsilon','gamma',
          'n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'selected_models.csv',selected_rows,
         ['outer_fold','procedure','estimator','representation','alpha','C','epsilon','gamma',
          'n_features','inner_pooled_RMSE_W_per_kg','source_run'])
save_csv(OUT/'held_out_predictions.csv',prediction_rows,
         ['procedure','estimator','representation','outer_fold','subject_id','trial_id','trial_uid',
          'observed_W_per_kg','predicted_W_per_kg','source_run'])
save_csv(OUT/'pooled_metrics.csv',pooled_metrics,
         ['procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'outer_fold_metrics.csv',fold_metrics,
         ['outer_fold','procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
save_csv(OUT/'paired_rmse_differences.csv',paired_diffs,
         ['representation','ridge_RMSE_W_per_kg','svr_RMSE_W_per_kg','ridge_minus_svr_RMSE_W_per_kg'])
save_csv(OUT/'outer_model_timings.csv',timing_rows,
         ['outer_fold','estimator','representation','alpha','C','epsilon','n_train_trials',
          'n_predicted_trials','fit_seconds','batch_prediction_seconds',
          'batch_amortised_seconds_per_trial','single_trial_prediction_total_seconds',
          'single_trial_prediction_mean_seconds','single_trial_prediction_median_seconds',
          'ridge_timing_refit_max_abs_difference_W_per_kg','run_type'])
save_csv(OUT/'timing_only_ridge_refit_checks.csv',timing_prediction_difference,
         ['outer_fold','representation','max_abs_difference_W_per_kg'])
save_csv(OUT/'verification_checks.csv',checks,['check','status'])

summary={
    'status':'PASS' if all(r['status']=='PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment':'E4.1','investigation':'I4','source_hdf5':H5.name,
    'source_sha256':source_sha,'reference_notebook04_zip':REFERENCE.name,
    'reference_notebook04_zip_sha256':reference_sha,
    'reference_notebook04_code_sha256':ref_summary.get('notebook_code_sha256'),
    'n_primary_trials':284,'n_participants':21,'n_primary_cycles':60757,
    'target':'stored corrected trial-level metcost_mean; net W/kg',
    'prediction_setting':'five fixed participant-disjoint outer folds',
    'evaluation_status':'internally held-out; previously inspected fixed folds',
    'inner_partitions':'exact saved Notebook 04 four grouped inner folds per outer fold',
    'input_representations':{'W-ALL9':909,'D-ALL9':54},
    'ridge_alphas':list(ALPHAS),'svr_C_values':list(C_GRID),
    'svr_epsilon_W_per_kg':list(EPSILON_GRID),'svr_kernel':'rbf','svr_gamma':'scale',
    'model_specification':'StandardScaler fitted on each training partition, then Ridge or RBF-SVR',
    'selection':'pooled trial-weighted inner RMSE; SVR exact ties smaller C, then larger epsilon',
    'reused_ridge_predictions':'unchanged Notebook 04 fixed W/D predictions',
    'comparisons':'Ridge minus SVR RMSE separately for W and D; positive favours SVR',
    'descriptive_W_minus_D_improvement_difference_W_per_kg':interaction,
    'timing_method':{
        'fit':'wall-clock perf_counter around fit of complete pipeline on each outer-development fold',
        'batch_prediction':'wall-clock perf_counter around prediction of the complete outer-test array',
        'batch_amortised_per_trial':'batch prediction elapsed divided by number of outer-test trials',
        'single_trial':'sequential predict calls for one held-out row; average and median per fold',
        'timing_only_ridge_refits':'one refit per W/D and fold; predictions checked but never replace Notebook 04',
        'scope':'feature extraction, source I/O, and hyperparameter search excluded from outer fit timings',
        'note':'wall-clock timings are environment-dependent descriptive measurements, not repeatability pass criteria',
    },
    'timing_sample_count':len(timing_rows),
    'started_at_utc':started_at,'finished_at_utc':datetime.now(timezone.utc).isoformat(),
    'model_evaluation_elapsed_seconds':experiment_seconds,
    'software':{
        'python':platform.python_version(),'numpy':np.__version__,
        'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,
        'h5py':h5py.__version__,'os':platform.platform(),
        'cpu_count':os.cpu_count(),'kernel_python_executable':sys.executable,
        'cpu':platform.processor(),
        'blas_threadpools':threadpool_info(),
        'thread_environment':{k:os.environ.get(k) for k in
            ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS')},
    },
    'notebook_filename':notebook_file.name,'notebook_code_sha256':notebook_code_sha,
    'code_hash_basis':'concatenated code-cell source UTF-8',
    'bootstrap_included':False,'repeatability_RC3_included':False,
    'verification':{r['check']:r['status'] for r in checks}
}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
with zipfile.ZipFile(OUTPUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(OUT.iterdir()):
        if file.is_file():
            archive.write(file,arcname=file.name)
print('Saved',len(list(OUT.iterdir())),'evidence files in',OUTPUT_ZIP)
print('Overall verification status:',summary['status'])